In [1]:
!pip install qdrant-client sentence-transformers openai


In [2]:
import os
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from google.colab import userdata

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [3]:
# Qdrant 클라이언트 초기화
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)


In [4]:
# 임베딩 모델 초기화
encoder = SentenceTransformer("jhgan/ko-sroberta-multitask")

# OpenAI 클라이언트 초기화
openai_api_key = userdata.get('FINAL_TEAM3')
if not openai_api_key:
    raise ValueError("OPENAI_API_KEY가 설정되지 않았습니다. Google Colab의 Secrets에서 'OPENAI_API_KEY'를 설정해주세요.")

openai_client = OpenAI(api_key=openai_api_key)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
def search_disease_info(query_text, limit=3):
    # 사용자가 입력한 질병 관련 텍스트를 인코딩하여 벡터로 변환
    query_vector = encoder.encode(query_text).tolist()

    # 검색 클라이언트를 사용하여 지정된 컬렉션('son5')에서 질병 정보를 검색
    search_results = client.search(
        collection_name='son5',  # 검색할 컬렉션 이름
        query_vector=query_vector,  # 검색할 질병의 벡터
        limit=limit  # 검색 결과의 최대 개수
    )

    results = []  # 검색 결과를 저장할 리스트 초기화
    for hit in search_results:  # 검색 결과를 반복
        result = {
            "질병": hit.payload.get("질병", "N/A"),  # 검색 결과에서 질병 이름 가져오기
            "답변": hit.payload.get("답변", "N/A"),  # 검색 결과에서 답변 가져오기
            "유사도 점수": hit.score  # 검색 결과의 유사도 점수 가져오기
        }
        results.append(result)  # 결과 리스트에 추가

    return results  # 최종 검색 결과 리스트 반환

In [6]:
def generate_response(query, context):
    # 사용자의 질문과 주어진 컨텍스트를 포함한 프롬프트 생성
    prompt = f"""당신은 의료 전문가입니다. 주어진 정보를 바탕으로 사용자의 질문에 정확하고 유용한 답변을 제공해주세요.

질문: {query}  # 사용자가 입력한 질문

컨텍스트: {context}  # 검색된 답변을 포함한 컨텍스트

답변:"""  # 답변을 작성할 부분

    # OpenAI 클라이언트를 사용하여 GPT 모델로부터 응답 생성
    response = openai_client.chat.completions.create(
        model="gpt-3.5-turbo",  # 사용하고자 하는 모델
        messages=[
            {"role": "system", "content": "You are a helpful medical assistant."},  # 시스템 메시지
            {"role": "user", "content": prompt}  # 사용자 메시지
        ],
        max_tokens=500,  # 생성할 최대 토큰 수
        n=1,  # 생성할 응답의 개수
        temperature=0.7,  # 응답의 창의성 조절 (0.0~1.0)
    )

    return response.choices[0].message.content.strip()  # 생성된 응답 반환

In [7]:
def advanced_rag_search():
    # 무한 루프를 시작하여 사용자로부터 질문을 입력받음
    while True:
        query = input("질문을 입력하세요 (종료하려면 '끝내기' 입력): ")  # 사용자 질문 입력
        if query.lower() == '끝내기':  # '끝내기' 입력 시 루프 종료
            print("검색을 종료합니다.")
            break

        # 사용자가 입력한 질문에 대한 질병 정보 검색
        results = search_disease_info(query)
        if results:  # 검색 결과가 있는 경우
            best_match = results[0]  # 가장 유사도가 높은 결과 선택
            context = best_match['답변']  # 선택된 결과의 답변을 컨텍스트로 사용

            # 검색 결과 출력
            print(f"\n'{query}'에 대한 검색 결과:\n")
            print(f"관련 질병: {best_match['질병']}")  # 관련 질병 출력
            print(f"유사도 점수: {best_match['유사도 점수']:.4f}")  # 유사도 점수 출력
            print("원본 답변:")
            print(context)  # 원본 답변 출력
            print("\nOpenAI GPT 생성 답변:")
            generated_response = generate_response(query, context)  # GPT 응답 생성
            print(generated_response)  # 생성된 응답 출력
        else:
            print("검색 결과가 없습니다.")  # 검색 결과가 없을 경우 메시지 출력
        print("-" * 50)  # 구분선 출력

# 실행
advanced_rag_search()  # 고급 RAG 검색 함수 호출

질문을 입력하세요 (종료하려면 '끝내기' 입력): 운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나

'운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나'에 대한 검색 결과:

관련 질병: 아킬레스 건염
유사도 점수: 0.8145
원본 답변:
아킬레스 건염은 주로 발목 부근에서 발생하는 염증성 질환으로 알려져 있습니다. 아킬레스 건염은 발목 주변에서 통증과 부종을 동반하는 통증이 주로 나타납니다. 운동과 같은 활동 후 발뒤꿈치의 아킬레스 건 부분에서 염증이 발생하면 발목 주변에 강한 통증이 생길 수 있습니다. 또한, 아킬레스 건염은 종아리로 통증이 전해질 수 있어 발목 부분을 걷거나 달리는 동안 통증이 느껴질 수 있습니다. 만약 아킬레스 건염을 의심한다면, 휴식을 취해도 통증이 지속되는 경우 의사의 진료를 받는 것이 중요합니다. 아킬레스 건염은 다양한 증상을 동반하는 염증성 질환입니다. 만약 발목 주변에서 통증이 나타나면 의사의 진료를 받아야 합니다. 조기에 대처하지 않으면 심각한 합병증을 초래할 수 있으므로, 가능한 빠른 시일 내에 증상을 인지하고 의사의 도움을 받는 것이 중요합니다.

OpenAI GPT 생성 답변:
운동 전후 종아리 뒤쪽 통증이 발생하고 발목을 움직일 때 소리가 난다면 아킬레스 건염이 의심됩니다. 아킬레스 건염은 발목 주변에서 발생하는 염증으로, 통증과 부종을 동반할 수 있습니다. 운동이나 활동 후 발뒤꿈치 부분에서 염증이 발생하여 발목 주변에 강한 통증이 나타날 수 있습니다. 만약 휴식을 취해도 통증이 지속된다면 의사의 진료를 받는 것이 중요합니다. 아킬레스 건염은 조기 대처가 필요한 질환으로 심각한 합병증을 초래할 수 있으므로 빠른 시일 내에 전문가의 도움을 받는 것이 중요합니다.
--------------------------------------------------
질문을 입력하세요 (종료하려면 '끝내기' 입력): 통풍 예방 방법이 있을까?

'통풍 예방 방법이 있을까?'에 대한 검색 결과:

관련 질병:

openAI gpt-3.5-turbo 사용 결과 :
 - 답변 깔끔하게 잘 나옴.
 - 답변 생성 시간 굉장히 빠름(1~2초안에 생성)
 - DB 답변 기반으로 텍스트를 생성함

결론 : 오픈 소스모델로 이정도 퀄리티를 낼 수 있을까?...
